# Chapter 3 — Linked Lists

**DS/MLE perspective first:** `collections.deque`, LRU caches, streaming pipelines — all linked list structures under the hood. This chapter shows the patterns, then the LeetCode problems that test them from scratch.

Two patterns, same order as `notes.md`:
1. **Fast & Slow Pointers** → streaming midpoint, cycle detection — LC 876, LC 141
2. **In-Place Reversal & Merge** → pointer manipulation, sorted merge — LC 206, LC 21

Each section: DS/ML version first → mechanism → LeetCode problem → side-by-side implementations → step-by-step trace.

In [12]:
from collections import deque

# ── ListNode definition and helpers ───────────────────────────────────────────

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def make_list(vals):
    """Create a linked list from a Python list. Returns head."""
    if not vals: return None
    head = ListNode(vals[0])
    cur = head
    for v in vals[1:]:
        cur.next = ListNode(v)
        cur = cur.next
    return head

def list_to_py(head):
    """Convert linked list to Python list."""
    result, cur = [], head
    while cur:
        result.append(cur.val)
        cur = cur.next
    return result

def show_list(head, markers=None):
    """
    Print linked list as: val1 → val2 → ... → None
    markers: dict of {node_index: label} to annotate positions, e.g. {2: 'slow', 4: 'fast'}
    """
    parts, cur, i = [], head, 0
    while cur:
        label = f"[{markers[i]}]" if markers and i in markers else ""
        parts.append(f"{cur.val}{label}")
        cur = cur.next
        i += 1
    print("  " + " → ".join(parts) + " → None")

---
## Part 1 — Fast & Slow Pointers

### What you already use: `deque` and streaming pipelines

`collections.deque` is Python's doubly linked list. When you process a data stream of unknown length and need the middle element, you either buffer everything (O(n) space) or use fast/slow pointers (O(1) space).

In [13]:
# Simulating a data stream of unknown length
import random
random.seed(42)
stream = [random.randint(1, 100) for _ in range(11)]

# Naive: buffer everything, then index
buffered = list(stream)
mid_buffered = buffered[len(buffered) // 2]
print(f"Stream: {stream}")
print(f"Middle (buffered, O(n) space): index {len(buffered)//2} → value {mid_buffered}")
print()
print("Fast/slow pointer approach uses O(1) space — no buffer needed.")
print("This matters for truly streaming data where you can't store the full sequence.")

Stream: [82, 15, 4, 95, 36, 32, 29, 18, 95, 14, 87]
Middle (buffered, O(n) space): index 5 → value 32

Fast/slow pointer approach uses O(1) space — no buffer needed.
This matters for truly streaming data where you can't store the full sequence.


### Under the hood: the fast/slow mechanism

- **slow** advances 1 step per iteration
- **fast** advances 2 steps per iteration

When fast reaches the end, slow is exactly at the middle. If there's a cycle, fast eventually laps slow and they meet — proving a cycle exists.

```
Step 0: slow=1, fast=1
Step 1: slow=2, fast=3
Step 2: slow=3, fast=5  ← fast at end → slow is middle
```

---
### LC 876 — Middle of the Linked List

> Given the head of a singly linked list, return the middle node. If two middle nodes exist, return the second.
>
> Example: `[1,2,3,4,5]` → node with val `3`; `[1,2,3,4,5,6]` → node with val `4`

**DS parallel:** Finding the median of a streaming sequence without storing it. In pandas: `df.iloc[len(df)//2]` — but fast/slow works O(1) space on a stream where you don't know the length in advance.

**Key insight:** slow moves 1, fast moves 2. When `fast` or `fast.next` is None, `slow` is at the middle.

In [14]:
# ── PYTHON LIST APPROACH (O(n) space) ────────────────────────────────────────
def middleNode_list(head):
    vals = list_to_py(head)              # buffer everything
    mid_val = vals[len(vals) // 2]
    # find and return the node (for comparison)
    cur, i = head, 0
    while i < len(vals) // 2:
        cur = cur.next
        i += 1
    return cur

# ── FAST/SLOW POINTERS (the interview answer) ────────────────────────────────
# O(1) space. No buffer. Works on true streams.
def middleNode(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow  # slow is at the middle

for vals in [[1,2,3,4,5], [1,2,3,4,5,6]]:
    head = make_list(vals)
    mid_list = middleNode_list(head).val
    mid_fast = middleNode(head).val
    print(f"List: {vals}  →  list approach: {mid_list},  fast/slow: {mid_fast}")

List: [1, 2, 3, 4, 5]  →  list approach: 3,  fast/slow: 3
List: [1, 2, 3, 4, 5, 6]  →  list approach: 4,  fast/slow: 4


In [15]:
# ── Step-by-step trace with pointer visualization ─────────────────────────────
vals = [1, 2, 3, 4, 5]
head = make_list(vals)
print(f"List: {vals}")
print()

# Build index→node map for show_list
nodes = []
cur = head
while cur:
    nodes.append(cur)
    cur = cur.next

def node_idx(node):
    for i, n in enumerate(nodes):
        if n is node: return i
    return -1

slow = fast = head
step = 0
print(f"Step {step}: slow={slow.val} (idx {node_idx(slow)}), fast={fast.val} (idx {node_idx(fast)})")
show_list(head, markers={node_idx(slow): 'S/F'})
print()

while fast and fast.next:
    slow = slow.next
    fast = fast.next.next
    step += 1
    si, fi = node_idx(slow), node_idx(fast) if fast else -1
    fast_label = f"{fast.val} (idx {fi})" if fast else "None"
    print(f"Step {step}: slow={slow.val} (idx {si}), fast={fast_label}")
    m = {si: 'S'}
    if fast: m[fi] = 'F'
    show_list(head, markers=m)
    print()

print(f"fast is None or fast.next is None → slow is at the middle: {slow.val}")

List: [1, 2, 3, 4, 5]

Step 0: slow=1 (idx 0), fast=1 (idx 0)
  1[S/F] → 2 → 3 → 4 → 5 → None

Step 1: slow=2 (idx 1), fast=3 (idx 2)
  1 → 2[S] → 3[F] → 4 → 5 → None

Step 2: slow=3 (idx 2), fast=5 (idx 4)
  1 → 2 → 3[S] → 4 → 5[F] → None

fast is None or fast.next is None → slow is at the middle: 3


---
### LC 141 — Linked List Cycle

> Given a linked list, return `True` if it contains a cycle.

**DS parallel:** Detecting circular dependencies in Airflow DAG definitions, circular imports in Python packages, or cycles in a feature engineering pipeline graph.

**Key insight:** In a cycle, fast eventually laps slow. If fast reaches `None`, there's no cycle. Floyd's cycle detection — O(1) space vs O(n) for a visited set.

In [16]:
# ── VISITED SET (O(n) space) ──────────────────────────────────────────────────
def hasCycle_set(head):
    seen = set()
    cur = head
    while cur:
        if id(cur) in seen:
            return True
        seen.add(id(cur))
        cur = cur.next
    return False

# ── FAST/SLOW (Floyd's cycle detection, O(1) space) ───────────────────────────
def hasCycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:   # same node object — cycle!
            return True
    return False

# Test on a list without a cycle
head_no_cycle = make_list([3, 2, 0, -4])
print(f"No cycle:  set={hasCycle_set(head_no_cycle)},  fast/slow={hasCycle(head_no_cycle)}")

# Build a cycle manually: -4 → 2 (index 1)
head_cycle = make_list([3, 2, 0, -4])
tail = head_cycle
cycle_target = head_cycle.next   # node with val=2
while tail.next:
    tail = tail.next
tail.next = cycle_target         # create cycle: -4 → 2
print(f"With cycle: set={hasCycle_set(head_cycle)},  fast/slow={hasCycle(head_cycle)}")

No cycle:  set=False,  fast/slow=False
With cycle: set=True,  fast/slow=True


In [17]:
# ── Step-by-step trace on a cycle ─────────────────────────────────────────────
# Build list [1, 2, 3, 4] with 4 → 2 (cycle)
nodes_c = [ListNode(v) for v in [1, 2, 3, 4]]
for i in range(len(nodes_c) - 1):
    nodes_c[i].next = nodes_c[i + 1]
nodes_c[-1].next = nodes_c[1]   # 4 → 2 (index 1)
head_c = nodes_c[0]

print("List: 1 → 2 → 3 → 4 → (back to 2)  [cycle at index 1]")
print()

def node_name(node, nodes_list):
    for i, n in enumerate(nodes_list):
        if n is node: return f"{n.val}(idx {i})"
    return "None"

slow = fast = head_c
step = 0
print(f"  {'step':>4}  {'slow':>12}  {'fast':>12}  {'slow is fast?':>14}")
print("  " + "-" * 50)
print(f"  {step:>4}  {node_name(slow, nodes_c):>12}  {node_name(fast, nodes_c):>12}  {str(slow is fast):>14}")

for _ in range(8):  # enough steps to show cycle detection
    if not (fast and fast.next): break
    slow = slow.next
    fast = fast.next.next
    step += 1
    meeting = slow is fast
    print(f"  {step:>4}  {node_name(slow, nodes_c):>12}  {node_name(fast, nodes_c):>12}  {str(meeting):>14}")
    if meeting:
        print(f"\n  → slow and fast met at {node_name(slow, nodes_c)} — cycle detected!")
        break

List: 1 → 2 → 3 → 4 → (back to 2)  [cycle at index 1]

  step          slow          fast   slow is fast?
  --------------------------------------------------
     0      1(idx 0)      1(idx 0)            True
     1      2(idx 1)      3(idx 2)           False
     2      3(idx 2)      2(idx 1)           False
     3      4(idx 3)      4(idx 3)            True

  → slow and fast met at 4(idx 3) — cycle detected!


---
## Part 2 — In-Place Reversal & Merge

### What you already use: `list[::-1]` and `pd.merge_asof`

Reversing a list in Python creates a new copy. Reversing a linked list in-place is O(1) space — each node's `.next` pointer is redirected. Merging two sorted linked lists is the same two-pointer merge that `pd.merge_asof` uses on sorted timestamps.

In [18]:
import pandas as pd
import numpy as np

# pd.merge_asof is a two-pointer merge on sorted keys
prices = pd.DataFrame({'time': [1, 3, 5, 7], 'price': [100, 102, 99, 105]})
events = pd.DataFrame({'time': [2, 6],       'event': ['earnings', 'split']})
merged = pd.merge_asof(events, prices, on='time')
print("pd.merge_asof — two-pointer merge under the hood:")
print(merged.to_string(index=False))
print()
print("LC 21 (Merge Two Sorted Lists) is this exact algorithm applied to linked list nodes.")

pd.merge_asof — two-pointer merge under the hood:
 time    event  price
    2 earnings    100
    6    split     99

LC 21 (Merge Two Sorted Lists) is this exact algorithm applied to linked list nodes.


### Under the hood: pointer manipulation

**Reversal:** Three pointers — `prev`, `curr`, `next_node`. At each step: save `next_node = curr.next`, flip `curr.next = prev`, advance `prev = curr`, advance `curr = next_node`.

**Merge:** Dummy head eliminates edge cases. Two pointers, one per list. Always attach the node with the smaller value. Advance that list's pointer.

The key mindset: you're not moving values — you're redirecting arrows (pointers).

---
### LC 206 — Reverse Linked List

> Reverse a singly linked list in-place. Return the new head.
>
> Example: `[1,2,3,4,5]` → `[5,4,3,2,1]`

**DS parallel:** `list[::-1]` — but that creates a new list (O(n) space). The linked list reversal is O(1) space by redirecting pointers. Also the mechanism behind deque rotation and certain in-place array algorithms.

**Key insight:** Three pointers. Save `next_node` before overwriting `curr.next`. At the end, `prev` is the new head.

In [19]:
# ── PYTHON LIST APPROACH (O(n) space, not in-place) ───────────────────────────
def reverseList_copy(head):
    vals = list_to_py(head)[::-1]    # creates reversed copy
    return make_list(vals)

# ── IN-PLACE ITERATIVE (the interview answer, O(1) space) ─────────────────────
def reverseList(head):
    prev, curr = None, head
    while curr:
        next_node  = curr.next    # 1. save next
        curr.next  = prev         # 2. flip pointer
        prev       = curr         # 3. advance prev
        curr       = next_node    # 4. advance curr
    return prev  # prev is now the new head

for vals in [[1,2,3,4,5], [1,2], [1]]:
    head = make_list(vals)
    rev_copy = list_to_py(reverseList_copy(head))
    head2 = make_list(vals)
    rev_inplace = list_to_py(reverseList(head2))
    print(f"{vals} → copy: {rev_copy},  in-place: {rev_inplace}  ✓" if rev_copy == rev_inplace else "✗")

[1, 2, 3, 4, 5] → copy: [5, 4, 3, 2, 1],  in-place: [5, 4, 3, 2, 1]  ✓
[1, 2] → copy: [2, 1],  in-place: [2, 1]  ✓
[1] → copy: [1],  in-place: [1]  ✓


In [20]:
# ── Step-by-step trace with pointer state at each step ────────────────────────
vals = [1, 2, 3, 4, 5]
head = make_list(vals)
print(f"Input: {vals}")
print()

# Show initial state
prev, curr = None, head
step = 0
print(f"Initial state:")
print(f"  prev=None,  curr={curr.val}")
show_list(head)
print()

# Collect nodes for display
all_nodes = []
c = head
while c:
    all_nodes.append(c)
    c = c.next

while curr:
    next_node = curr.next
    curr.next = prev
    prev = curr
    curr = next_node
    step += 1

    prev_val = prev.val if prev else None
    curr_val = curr.val if curr else None
    next_val = next_node.val if next_node else None

    print(f"Step {step}: flip {prev_val}→ back,  prev={prev_val},  curr={curr_val},  saved_next={next_val}")
    # Show current reversed portion
    rev_so_far = []
    p = prev
    while p:
        rev_so_far.append(p.val)
        p = p.next
    print(f"  Reversed so far: {rev_so_far}  |  Remaining: {[n.val for n in all_nodes if n is curr or (curr and n.val in list_to_py(curr))]}")
    print()

print(f"Result: {list_to_py(prev)}")

Input: [1, 2, 3, 4, 5]

Initial state:
  prev=None,  curr=1
  1 → 2 → 3 → 4 → 5 → None

Step 1: flip 1→ back,  prev=1,  curr=2,  saved_next=2
  Reversed so far: [1]  |  Remaining: [2, 3, 4, 5]

Step 2: flip 2→ back,  prev=2,  curr=3,  saved_next=3
  Reversed so far: [2, 1]  |  Remaining: [3, 4, 5]

Step 3: flip 3→ back,  prev=3,  curr=4,  saved_next=4
  Reversed so far: [3, 2, 1]  |  Remaining: [4, 5]

Step 4: flip 4→ back,  prev=4,  curr=5,  saved_next=5
  Reversed so far: [4, 3, 2, 1]  |  Remaining: [5]

Step 5: flip 5→ back,  prev=5,  curr=None,  saved_next=None
  Reversed so far: [5, 4, 3, 2, 1]  |  Remaining: []

Result: [5, 4, 3, 2, 1]


---
### LC 21 — Merge Two Sorted Lists

> Merge two sorted linked lists into one sorted list. Return the head of the merged list.
>
> Example: `l1 = [1,2,4], l2 = [1,3,4]` → `[1,1,2,3,4]`

**DS parallel:** The core step of merge sort; `pd.merge_asof` on sorted timestamps; merging two sorted event streams. Same two-pointer pattern from Chapter 1 applied to linked list nodes.

**Key insight:** Dummy head eliminates edge cases when attaching the first node. Two pointers advance through each list; always take the node with the smaller value.

In [21]:
# ── HEAPQ MERGE (what you'd use in Python for sorted iterables) ───────────────
import heapq
def mergeLists_heapq(l1, l2):
    vals1, vals2 = list_to_py(l1), list_to_py(l2)
    return make_list(list(heapq.merge(vals1, vals2)))

# ── DUMMY HEAD + TWO POINTERS (the interview answer) ─────────────────────────
# O(n + m) time, O(1) extra space.
def mergeTwoLists(list1, list2):
    dummy = ListNode(0)      # sentinel — avoids special-casing the head
    cur = dummy
    l1, l2 = list1, list2
    while l1 and l2:
        if l1.val <= l2.val:
            cur.next = l1
            l1 = l1.next
        else:
            cur.next = l2
            l2 = l2.next
        cur = cur.next
    cur.next = l1 or l2      # attach remaining nodes
    return dummy.next

test_cases = [
    ([1,2,4], [1,3,4], [1,1,2,3,4]),
    ([],      [0],     [0]),
    ([],      [],      []),
]
for v1, v2, expected in test_cases:
    h1, h2 = make_list(v1), make_list(v2)
    h1b, h2b = make_list(v1), make_list(v2)
    hq_res  = list_to_py(mergeLists_heapq(h1, h2))
    tp_res  = list_to_py(mergeTwoLists(h1b, h2b))
    match = "✓" if tp_res == expected else "✗"
    print(f"{v1} + {v2} → heapq: {hq_res},  two-ptr: {tp_res}  {match}")

[1, 2, 4] + [1, 3, 4] → heapq: [1, 1, 2, 3, 4, 4],  two-ptr: [1, 1, 2, 3, 4, 4]  ✗
[] + [0] → heapq: [0],  two-ptr: [0]  ✓
[] + [] → heapq: [],  two-ptr: []  ✓


In [22]:
# ── Step-by-step trace — shows which node is chosen at each step ──────────────
v1, v2 = [1, 2, 4], [1, 3, 4]
l1, l2 = make_list(v1), make_list(v2)
print(f"l1: {v1}")
print(f"l2: {v2}")
print()

dummy = ListNode(0)
cur = dummy
step = 0

print(f"  {'step':>4}  {'l1 head':>8}  {'l2 head':>8}  {'chosen':>8}  merged so far")
print("  " + "-" * 55)

while l1 and l2:
    l1_val = l1.val if l1 else None
    l2_val = l2.val if l2 else None
    if l1.val <= l2.val:
        chosen = l1.val
        cur.next = l1
        l1 = l1.next
        choice = f"l1({chosen})"
    else:
        chosen = l2.val
        cur.next = l2
        l2 = l2.next
        choice = f"l2({chosen})"
    cur = cur.next
    step += 1
    merged_so_far = list_to_py(dummy.next)
    l1_rem = l1.val if l1 else "None"
    l2_rem = l2.val if l2 else "None"
    print(f"  {step:>4}  {str(l1_val):>8}  {str(l2_val):>8}  {choice:>10}  {merged_so_far}")

cur.next = l1 or l2
print(f"\n  Remaining list attached. Final: {list_to_py(dummy.next)}")

l1: [1, 2, 4]
l2: [1, 3, 4]

  step   l1 head   l2 head    chosen  merged so far
  -------------------------------------------------------
     1         1         1       l1(1)  [1, 2, 4]
     2         2         1       l2(1)  [1, 1, 3, 4]
     3         2         3       l1(2)  [1, 1, 2, 4]
     4         4         3       l2(3)  [1, 1, 2, 3, 4]
     5         4         4       l1(4)  [1, 1, 2, 3, 4]

  Remaining list attached. Final: [1, 1, 2, 3, 4, 4]


---
## Summary

| Pattern | LeetCode | DS/ML Equivalent | Time | Space |
|---------|----------|------------------|------|-------|
| Fast/slow — middle | LC 876 Middle of Linked List | `df.iloc[len(df)//2]` but streaming | O(n) | O(1) |
| Fast/slow — cycle | LC 141 Linked List Cycle | Circular dependency detection | O(n) | O(1) |
| In-place reversal | LC 206 Reverse Linked List | `list[::-1]` but O(1) space | O(n) | O(1) |
| Dummy head + two-ptr merge | LC 21 Merge Two Sorted Lists | `pd.merge_asof` / merge sort step | O(n+m) | O(1) |

**The core mindset:** You're not moving values — you're redirecting arrows. Draw out a small example (3-4 nodes) and physically trace where each pointer points before and after each step. This is the most reliable way to debug pointer manipulation problems.